# 🫀 CardioPredict: Heart Disease Prediction and Analytics
### Introduction to Data Science (IDS) Capstone Project
---
## Phase 1: Project Setup & Data Loading
**Dataset**: UCI Heart Disease Repository (Cleveland Clinic Foundation)  
**Objective**: Load raw data, configure attributes, binarize target, and inspect data types.

In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", "{:.2f}".format)
print("Libraries loaded successfully.")

In [ ]:
dataset_path = os.path.join("..", "dataset", "processed.cleveland.data")
column_names = [
    "Age", "Sex", "ChestPainType", "RestingBP", "Cholesterol",
    "FastingBS", "RestingECG", "MaxHR", "ExerciseAngina",
    "Oldpeak", "ST_Slope", "CA", "Thal", "Target"
]

heart_df = pd.read_csv(dataset_path, header=None, names=column_names, na_values="?")
heart_df["Target"] = heart_df["Target"].apply(lambda x: 1 if x > 0 else 0)
print(f"Dataset shape: {heart_df.shape}")

---
## Phase 2: Complete Data Cleaning & Preprocessing

In [ ]:
ca_mode = heart_df["CA"].mode()[0]
thal_mode = heart_df["Thal"].mode()[0]

heart_df["CA"] = heart_df["CA"].fillna(ca_mode)
heart_df["Thal"] = heart_df["Thal"].fillna(thal_mode)

int_cols = [
    "Age", "Sex", "ChestPainType", "RestingBP", "Cholesterol",
    "FastingBS", "RestingECG", "MaxHR", "ExerciseAngina",
    "ST_Slope", "CA", "Thal", "Target"
]
for col in int_cols:
    heart_df[col] = heart_df[col].astype("int64")

cleaned_path = os.path.join("..", "dataset", "heart_cleaned.csv")
heart_df.to_csv(cleaned_path, index=False)
print(f"Cleaned dataset saved: {heart_df.shape}")

---
## Phase 3: Descriptive Statistics & Exploratory Data Analysis (EDA)

In [ ]:
num_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
summary_metrics = pd.DataFrame({
    "Mean": heart_df[num_cols].mean(),
    "Median": heart_df[num_cols].median(),
    "Mode": [heart_df[col].mode()[0] for col in num_cols],
    "Min": heart_df[num_cols].min(),
    "Max": heart_df[num_cols].max(),
    "Range": heart_df[num_cols].max() - heart_df[num_cols].min(),
    "Variance": heart_df[num_cols].var(),
    "Std Dev": heart_df[num_cols].std(),
    "Q1 (25%)": heart_df[num_cols].quantile(0.25),
    "Q3 (75%)": heart_df[num_cols].quantile(0.75),
    "IQR": heart_df[num_cols].quantile(0.75) - heart_df[num_cols].quantile(0.25),
    "Skewness": heart_df[num_cols].skew(),
    "Kurtosis": heart_df[num_cols].kurtosis()
})
summary_metrics.round(2)

---
## Phase 4: Data Visualization & Exploratory Graphical Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

palette_colors = ["#2ecc71", "#e74c3c"]
plt.figure(figsize=(9, 5))
sns.histplot(data=heart_df, x="Age", hue="Target", bins=16, kde=True, palette=palette_colors, multiple="stack")
plt.title("Age Distribution Stratified by Diagnosis")
plt.show()

---
## Phase 5: Inferential Statistics & Hypothesis Testing

In [ ]:
from scipy import stats

# 1. Independent T-Test (Cholesterol vs Target)
chol_healthy = heart_df[heart_df["Target"] == 0]["Cholesterol"]
chol_disease = heart_df[heart_df["Target"] == 1]["Cholesterol"]
t_stat, t_pval = stats.ttest_ind(chol_healthy, chol_disease, equal_var=True)
print(f"T-Test: t = {t_stat:.4f}, p = {t_pval:.4f}")

# 2. Chi-Square Test (Sex vs Target)
observed = pd.crosstab(heart_df["Sex"], heart_df["Target"])
chi2_stat, chi2_pval, dof, expected = stats.chi2_contingency(observed)
print(f"Chi2: statistic = {chi2_stat:.4f}, p = {chi2_pval:.4e}, df = {dof}")

---
## Phase 6: Data Preprocessing & Pipeline Preparation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = heart_df.drop(columns=["Target"])
y = heart_df["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train size: {X_train_scaled.shape}, Test size: {X_test_scaled.shape}")

---
## Phase 7: Machine Learning Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=4),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6),
    "SVM": SVC(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=7)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{name:<22}: Train Acc = {train_acc*100:.2f}%, Test Acc = {test_acc*100:.2f}%")

---
## Phase 8: Comprehensive Model Evaluation & Benchmarking

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

eval_rows = []
for name, model in models.items():
    preds = model.predict(X_test_scaled)
    scores = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_test_scaled)
    
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    
    eval_rows.append({
        'Algorithm': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall (Sensitivity)': recall_score(y_test, preds),
        'Specificity': tn / (tn + fp),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, scores)
    })

eval_df = pd.DataFrame(eval_rows)
eval_df.round(4)